<h1 align="center">Join Submissions, Numeric and Tags Data Sets to Find Custom Facts</h1>

### Import Code Libraries

In [ ]:
import polars as pl
from IPython.display import display, HTML

display(HTML("<style>.container { width:100% !important; }</style>"))

### Read Submissions, Numeric and Tags Data Sets for Feb. 2024 from [Financial Statement and Notes Data Sets](https://www.sec.gov/dera/data/financial-statement-and-notes-data-set)

In [ ]:
submissions = pl.read_csv('data/2024q1_notes/sub.tsv', separator='\t', infer_schema_length=10000)
numericFacts = pl.read_csv('data/2024q1_notes/num.tsv', separator='\t', infer_schema_length=10000)
tags = pl.read_csv('data/2024q1_notes/tag.tsv', separator='\t', infer_schema_length=10000)

### Look at the Contents of submissions DataFrame

In [ ]:
submissions

### Look at the Contents of numericFacts DataFrame

In [ ]:
numericFacts

### Look at the Contents of tags DataFrame

In [ ]:
tags

### Filter for Custom Monetary Tags (abstract=0 means concrete, not abstract)

In [ ]:
customTagsDF = tags.filter(
    (pl.col('custom') == 1)
    & (pl.col('abstract') == 0)
    & (pl.col('datatype') == 'monetary')
)
customTagsDF

### Join submissions DataFrame with numericFacts DataFrame Using adsh Column as a Join Key

In [ ]:
mergedSubmissionsAndNumericFactsDF = submissions.join(numericFacts, on='adsh')

### Join with customTagsDF Using adsh and tag as Join Keys (version column = adsh for custom tags)

In [ ]:
mergedDF = mergedSubmissionsAndNumericFactsDF.join(
    customTagsDF,
    left_on=['adsh', 'tag'],
    right_on=['version', 'tag']
)

### Look at the Contents of mergedDF DataFrame

In [ ]:
mergedDF

### Filter for Facts with Custom Tags for AMAZON COM INC

In [ ]:
customFactsForAmazon = (
    mergedDF
    .filter(
        (pl.col('name') == 'AMAZON COM INC')
        & (pl.col('ddate') == 20231231)
        & (pl.col('dimn') == 0)
    )
    .select(['name', 'tag', 'dimn', 'ddate', 'qtrs', 'uom', 'value'])
    .sort('tag')
)
customFactsForAmazon